In [0]:
# =========================
# BRONZE -> CLEAN TEMP VIEWS (from 01_silver logic)
# =========================
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col, to_timestamp

BRONZE_PATH = "abfss://bronze@financestoragebablu.dfs.core.windows.net"

spark.read.parquet(f"{BRONZE_PATH}/stock").select("stock_id","ticker","company_name","sector","exchange",F.col("base_price").cast("double")).dropDuplicates(["stock_id"]).createOrReplaceTempView("v_stocks")
spark.read.parquet(f"{BRONZE_PATH}/market_event").select("event_id","stock_id","ticker",F.to_date("date").alias("event_date"),"event_type","impact",F.col("price_change_pct").cast("double"),"description").dropDuplicates(["event_id"]).createOrReplaceTempView("v_market_events")
spark.read.parquet(f"{BRONZE_PATH}/portfolio").select("portfolio_id","user_id","stock_id","ticker",F.col("quantity").cast("int"),F.col("avg_cost").cast("double"),F.col("total_invested").cast("double"),"status").dropDuplicates(["portfolio_id"]).createOrReplaceTempView("v_portfolios")
spark.read.parquet(f"{BRONZE_PATH}/price").select("price_id","stock_id","ticker",F.to_date("date").alias("price_date"),F.col("open").cast("double"),F.col("high").cast("double"),F.col("low").cast("double"),F.col("close").cast("double"),F.col("volume").cast("long")).dropDuplicates(["price_id"]).createOrReplaceTempView("v_prices")
spark.read.parquet(f"{BRONZE_PATH}/transaction").select("txn_id","portfolio_id","user_id","stock_id","ticker","txn_type",F.col("quantity").cast("int"),F.col("price").cast("double"),F.col("total_value").cast("double"),F.col("fee").cast("double"),F.to_date("date").alias("txn_date"),"status").dropDuplicates(["txn_id"]).createOrReplaceTempView("v_transactions")

# Event Hub
df_eh = spark.read.table("finance.bronze.stocktopic")
base_schema = StructType([StructField("event_type",StringType()),StructField("sequence_id",LongType()),StructField("timestamp",StringType())])
tick_schema = StructType([StructField("event_type",StringType()),StructField("sequence_id",LongType()),StructField("timestamp",StringType()),StructField("symbol",StringType()),StructField("price",DoubleType()),StructField("bid",DoubleType()),StructField("ask",DoubleType()),StructField("volume",LongType()),StructField("exchange",StringType())])
cdc_schema = StructType([StructField("event_type",StringType()),StructField("sequence_id",LongType()),StructField("timestamp",StringType()),StructField("db_table",StringType()),StructField("operation",StringType()),StructField("transaction_id",StringType()),StructField("before",StringType()),StructField("after",StringType()),StructField("source",StringType())])
ohlcv_schema = StructType([StructField("event_type",StringType()),StructField("sequence_id",LongType()),StructField("timestamp",StringType()),StructField("symbol",StringType()),StructField("interval",StringType()),StructField("open",DoubleType()),StructField("high",DoubleType()),StructField("low",DoubleType()),StructField("close",DoubleType()),StructField("volume",LongType()),StructField("exchange",StringType())])

df_eh.filter(from_json("value",base_schema)["event_type"]=="TICK").withColumn("d",from_json("value",tick_schema)).select(col("d.sequence_id"),to_timestamp("d.timestamp").alias("event_timestamp"),col("d.symbol"),col("d.price"),col("d.bid"),col("d.ask"),col("d.volume"),col("d.exchange")).dropDuplicates(["sequence_id"]).createOrReplaceTempView("v_tick")
df_eh.filter(from_json("value",base_schema)["event_type"]=="CDC").withColumn("d",from_json("value",cdc_schema)).select(col("d.sequence_id"),to_timestamp("d.timestamp").alias("event_timestamp"),col("d.db_table"),col("d.operation"),col("d.transaction_id"),col("d.before").alias("before_state"),col("d.after").alias("after_state"),col("d.source").alias("source_info")).dropDuplicates(["sequence_id"]).createOrReplaceTempView("v_cdc")
df_eh.filter(from_json("value",base_schema)["event_type"]=="OHLCV").withColumn("d",from_json("value",ohlcv_schema)).select(col("d.sequence_id"),to_timestamp("d.timestamp").alias("event_timestamp"),col("d.symbol"),col("d.interval"),col("d.open"),col("d.high"),col("d.low"),col("d.close"),col("d.volume"),col("d.exchange")).dropDuplicates(["sequence_id"]).createOrReplaceTempView("v_ohlcv")

print(" All 8 temp views loaded from bronze")

In [0]:
# =========================
# STAR SCHEMA CONFIG (Incremental MERGE into finance.silver)
# =========================
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col, year, month, dayofmonth, quarter, dayofweek,
    date_format, when, current_timestamp, current_date
)

CATALOG = "finance"
SCHEMA = "silver"
TARGET = f"{CATALOG}.{SCHEMA}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET}")

def upsert(df_source, table_name, merge_key):
    """Incremental upsert: INSERT new, UPDATE existing."""
    full_table = f"{TARGET}.{table_name}"
    df_source = df_source.withColumn("_loaded_at", current_timestamp())

    if spark.catalog.tableExists(full_table):
        dt = DeltaTable.forName(spark, full_table)
        dt.alias("t").merge(
            df_source.alias("s"),
            f"t.{merge_key} = s.{merge_key}"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
        action = "MERGED"
    else:
        df_source.write.format("delta").saveAsTable(full_table)
        action = "CREATED"

    count = spark.table(full_table).count()
    print(f"  \u2705 {full_table} \u2014 {count:,} rows ({action})")

print("\u2705 Star schema config ready (target: finance.silver)")

In [0]:
# =========================
# DIM_DATE — derived from all date columns in temp views
# =========================
dates_txn = spark.table("v_transactions").select(col("txn_date").alias("full_date"))
dates_price = spark.table("v_prices").select(col("price_date").alias("full_date"))
dates_event = spark.table("v_market_events").select(col("event_date").alias("full_date"))

df_dates = (
    dates_txn.union(dates_price).union(dates_event)
    .distinct()
    .filter(col("full_date").isNotNull())
    .withColumn("date_id", date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", year("full_date"))
    .withColumn("month", month("full_date"))
    .withColumn("day", dayofmonth("full_date"))
    .withColumn("quarter", quarter("full_date"))
    .withColumn("weekday", dayofweek("full_date"))
    .withColumn("day_name", date_format("full_date", "EEEE"))
    .withColumn("month_name", date_format("full_date", "MMMM"))
    .withColumn("is_weekend", when(dayofweek("full_date").isin(1, 7), True).otherwise(False))
    .select("date_id", "full_date", "year", "month", "day", "quarter",
            "weekday", "day_name", "month_name", "is_weekend")
)

print("DIM_DATE")
upsert(df_dates, "dim_date", "date_id")

In [0]:
# =========================
# DIM_STOCK
# =========================
df_dim_stock = spark.table("v_stocks")

print("DIM_STOCK")
upsert(df_dim_stock, "dim_stock", "stock_id")

In [0]:
# =========================
# DIM_USER — derived from portfolios + transactions
# =========================
users_portfolio = spark.table("v_portfolios").select("user_id").distinct()
users_txn = spark.table("v_transactions").select("user_id").distinct()

df_dim_user = (
    users_portfolio.union(users_txn)
    .distinct()
    .filter(col("user_id").isNotNull())
)

print("DIM_USER")
upsert(df_dim_user, "dim_user", "user_id")

In [0]:
# =========================
# DIM_PORTFOLIO
# =========================
df_dim_portfolio = (
    spark.table("v_portfolios")
    .select("portfolio_id", "user_id", "status")
    .distinct()
)

print("DIM_PORTFOLIO")
upsert(df_dim_portfolio, "dim_portfolio", "portfolio_id")

In [0]:
# =========================
# DIM_EVENT
# =========================
df_dim_event = (
    spark.table("v_market_events")
    .withColumn("date_id", date_format("event_date", "yyyyMMdd").cast("int"))
    .select("event_id", "stock_id", "date_id",
            "event_type", "impact", "price_change_pct", "description")
)

print("DIM_EVENT")
upsert(df_dim_event, "dim_event", "event_id")

In [0]:
# =========================
# FACT_TRANSACTIONS — Grain: One transaction per row
# =========================
df_fact_txn = (
    spark.table("v_transactions")
    .withColumn("date_id", date_format("txn_date", "yyyyMMdd").cast("int"))
    .select("txn_id", "portfolio_id", "user_id", "stock_id", "date_id",
            "txn_type", "quantity", "price", "total_value", "fee", "status")
)

print("FACT_TRANSACTIONS")
upsert(df_fact_txn, "fact_transactions", "txn_id")

In [0]:
# =========================
# FACT_PRICES — Grain: One stock per day
# =========================
df_fact_prices = (
    spark.table("v_prices")
    .withColumn("date_id", date_format("price_date", "yyyyMMdd").cast("int"))
    .select("price_id", "stock_id", "date_id",
            "open", "high", "low", "close", "volume")
)

print("FACT_PRICES")
upsert(df_fact_prices, "fact_prices", "price_id")

# =========================
# FACT_PORTFOLIO_SNAPSHOT — Grain: One holding per portfolio per stock
# =========================
df_fact_portfolio = (
    spark.table("v_portfolios")
    .withColumn("snapshot_date_id", date_format(current_date(), "yyyyMMdd").cast("int"))
    .select("portfolio_id", "user_id", "stock_id",
            "snapshot_date_id", "quantity", "avg_cost", "total_invested")
)

print("FACT_PORTFOLIO_SNAPSHOT")
upsert(df_fact_portfolio, "fact_portfolio_snapshot", "portfolio_id")

In [0]:
# =========================
# VERIFY: Star Schema in finance.silver
# =========================
print(f"{'='*55}")
print(f"{'STAR SCHEMA SUMMARY (finance.silver)':^55}")
print(f"{'='*55}")

tables = spark.sql(f"SHOW TABLES IN {TARGET}").collect()

dims = []
facts = []

for t in tables:
    name = t.tableName
    if name.startswith("v_") or name.startswith("_"):
        continue
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM {TARGET}.{name}").collect()[0].cnt
    if name.startswith("dim_"):
        dims.append((name, count))
    elif name.startswith("fact_"):
        facts.append((name, count))

print(f"\n\U0001f7e1 DIMENSION TABLES")
print(f"  {'Table':<30} {'Rows':>10}")
print(f"  {'-'*42}")
for name, cnt in sorted(dims):
    print(f"  {name:<30} {cnt:>10,}")

print(f"\n\U0001f7e2 FACT TABLES")
print(f"  {'Table':<30} {'Rows':>10}")
print(f"  {'-'*42}")
for name, cnt in sorted(facts):
    print(f"  {name:<30} {cnt:>10,}")

print(f"\n\u2705 Total: {len(dims)} dimensions + {len(facts)} facts = {len(dims)+len(facts)} tables")